In [251]:
import pandas as pd
import os
import re
import numpy as np

In [252]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input/tornado")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output/tornado")

In [253]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    df = df.copy()

    # Extrae el sector: lo que está entre TX: y el siguiente :
    # "Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ" -> "AGRC"
    df["sector"] = df[strategy_col].str.extract(r"TX:([A-Z]{3,6}):", expand=False)

    # Caso especial baseline
    #df.loc[df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # Extrae transformation_name: lo que está después de TX:SECTOR:
    # "Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ" -> "DEC_CH4_RICE_STRATEGY_NZ from NZ"
    df["transformation_name"] = (
    df[strategy_col]
    .str.extract(r"TX:[A-Z]{3,6}:(\S+)", expand=False)
    .str.replace(r"_STRATEGY_\w+$", "", regex=True)
    )
    
    # Si quieres solo hasta el espacio (sin "from NZ"), usa \S+ que ya captura hasta el primer espacio
    # Si quieres todo lo que sigue incluyendo "from NZ", cambia \S+ por (.+)

    # # Caso baseline
    # base_mask = df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False)
    # df.loc[base_mask, "transformation_name"] = "BASE"

    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [254]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "raw_emissions_uganda_2019_tornado_data_raw.csv"))
# emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_bulgaria_2022_trww_debug.csv"))
emissions_df.head()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.241921,2019,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
1,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.244762,2020,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
2,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252282,2021,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
3,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.251119,2022,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
4,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252083,2023,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE


In [255]:
print(emissions_df.primary_id.nunique())

59


In [256]:
print(emissions_df.primary_id.nunique())

59


In [257]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE',
       'Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:DEC_LOSSES_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FGTV:DEC_LEAKS_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FGTV:INC_FLARE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FRST:INCREASE_SEQUESTRATION_NZ to Strategy TX:BASE',
       'Add TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:IPPU:DEC_CLINKER_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:IPPU:DEC_HFCS_STRATEGY_NZ to Strat

In [258]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

60
59


In [259]:
filtered_emissions_df["strategy"].unique()

array(['Strategy TX:BASE',
       'Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:DEC_LOSSES_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FGTV:DEC_LEAKS_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FGTV:INC_FLARE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FRST:INCREASE_SEQUESTRATION_NZ to Strategy TX:BASE',
       'Add TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:IPPU:DEC_CLINKER_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:IPPU:DEC_HFCS_STRATEGY_NZ to Strat

In [260]:
filtered_emissions_df.tail()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
165667,6558.0,1340134.0,Wetlands:co2,Wetlands,LULUCF,0.0,2066,co2,0.0,0.0,Add TX:WASO:INC_RECYCLING_STRATEGY_NZ to Strat...,UGA,uganda,SISEPUEDE
165668,6558.0,1340134.0,Wetlands:co2,Wetlands,LULUCF,0.0,2067,co2,0.0,0.0,Add TX:WASO:INC_RECYCLING_STRATEGY_NZ to Strat...,UGA,uganda,SISEPUEDE
165669,6558.0,1340134.0,Wetlands:co2,Wetlands,LULUCF,0.0,2068,co2,0.0,0.0,Add TX:WASO:INC_RECYCLING_STRATEGY_NZ to Strat...,UGA,uganda,SISEPUEDE
165670,6558.0,1340134.0,Wetlands:co2,Wetlands,LULUCF,0.0,2069,co2,0.0,0.0,Add TX:WASO:INC_RECYCLING_STRATEGY_NZ to Strat...,UGA,uganda,SISEPUEDE
165671,6558.0,1340134.0,Wetlands:co2,Wetlands,LULUCF,0.0,2070,co2,0.0,0.0,Add TX:WASO:INC_RECYCLING_STRATEGY_NZ to Strat...,UGA,uganda,SISEPUEDE


In [261]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['Strategy TX:BASE',
       'Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:DEC_LOSSES_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FGTV:DEC_LEAKS_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FGTV:INC_FLARE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FRST:INCREASE_SEQUESTRATION_NZ to Strategy TX:BASE',
       'Add TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:IPPU:DEC_CLINKER_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:IPPU:DEC_HFCS_STRATEGY_NZ to Strat

In [262]:
tornado_emissions_df.head()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.241921,2019,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
1,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.244762,2020,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
2,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252282,2021,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
3,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.251119,2022,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
4,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252083,2023,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE


In [263]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,value
0,0.0,0.0,Strategy TX:BASE,10817.757318
1,6500.0,760076.0,Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strate...,10815.341657
2,6501.0,770077.0,Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_N...,10654.457050
3,6502.0,780078.0,Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRAT...,10793.617644
4,6503.0,790079.0,Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to St...,10495.590245


In [264]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,value
54,6554.0,1300130.0,Add TX:WASO:INC_CAPTURE_BIOGAS_STRATEGY_NZ to ...,10556.015150
55,6555.0,1310131.0,Add TX:WASO:INC_ENERGY_FROM_BIOGAS_STRATEGY_NZ...,10818.028541
56,6556.0,1320132.0,Add TX:WASO:INC_ENERGY_FROM_INCINERATION_STRAT...,10816.581018
57,6557.0,1330133.0,Add TX:WASO:INC_LANDFILLING_STRATEGY_NZ to Str...,11068.173466
58,6558.0,1340134.0,Add TX:WASO:INC_RECYCLING_STRATEGY_NZ to Strat...,10770.034739


In [265]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

59
59


In [266]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total column by setting it to the strategy_id == 0 value
base_emission_total = tornado_emissions_agg_df.loc[tornado_emissions_agg_df['strategy_id'] == 0, 'emission_total'].values[0]
tornado_emissions_agg_df['base_emission_total'] = base_emission_total

# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] =  tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']
tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_diff'].round(1)
tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff
0,0.0,0.0,Strategy TX:BASE,10817.757318,10817.757318,0.0
1,6500.0,760076.0,Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strate...,10815.341657,10817.757318,-2.4
2,6501.0,770077.0,Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_N...,10654.457050,10817.757318,-163.3
3,6502.0,780078.0,Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRAT...,10793.617644,10817.757318,-24.1
4,6503.0,790079.0,Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to St...,10495.590245,10817.757318,-322.2


In [267]:
tornado_emissions_agg_extended_df = add_sector_and_transformation_fields(tornado_emissions_agg_df)
tornado_emissions_agg_extended_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,0.0,0.0,Strategy TX:BASE,10817.757318,10817.757318,0.0,NaN,
1,6500.0,760076.0,Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strate...,10815.341657,10817.757318,-2.4,AGRC,DEC_CH4_RICE
2,6501.0,770077.0,Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_N...,10654.457050,10817.757318,-163.3,AGRC,DEC_LOSSES_SUPPLY_CHAIN
3,6502.0,780078.0,Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRAT...,10793.617644,10817.757318,-24.1,AGRC,INC_CONSERVATION_AGRICULTURE
4,6503.0,790079.0,Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to St...,10495.590245,10817.757318,-322.2,AGRC,INC_PRODUCTIVITY


In [268]:
tornado_emissions_agg_extended_df.to_clipboard(index=False)

## Load and process CB data

In [269]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "cba_results_ssp_modeling_tornado.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value
0,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
1,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
2,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
3,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
4,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0


In [270]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

# Split 'variable' into components: name, sector, cb_type, item_1, item_2
# (Assumes exactly 5 colon-separated parts; if there are more colons inside the last field,
# they will be kept in item_2 thanks to n=4)
cb_chars = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
cb_chars.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
cb_data = pd.concat([cb_data, cb_chars], axis=1)
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2
0,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
1,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
2,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
3,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
4,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural


In [271]:
# Scale value from USD to billions (divide by 1e9)
if "value" in cb_data.columns:
    cb_data["value"] = cb_data["value"] / 1e9

# --- Remove "shifted" entries ---
# # Remove rows where item_2 contains "shifted"
# cb_data = cb_data[~cb_data["item_2"].astype(str).str.contains("shifted", na=False)]

# # Remove any remaining rows where variable contains "shifted2"
# cb_data = cb_data[~cb_data["variable"].astype(str).str.contains("shifted2", na=False)]

# --- Add Year column (Year = time_period + 2015) ---
cb_data["Year"] = cb_data["time_period"] + 2015

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year
0,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019
1,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020
2,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021
3,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022
4,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023


In [272]:
# Load attribute strategy
attribute_strategy_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "ATTRIBUTE_STRATEGY.csv"))
attribute_strategy_df = attribute_strategy_df[["strategy_id", "strategy_code"]]
attribute_strategy_df.head()

,strategy_id,strategy_code
0,0,BASE
1,6500,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ
2,6501,TORNADO_BASE:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_S...
3,6502,TORNADO_BASE:TX:AGRC:INC_CONSERVATION_AGRICULT...
4,6503,TORNADO_BASE:TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ


In [273]:
attribute_strategy_df.strategy_id.unique()

array([   0, 6500, 6501, 6502, 6503, 6504, 6505, 6506, 6507, 6508, 6509,
       6510, 6511, 6512, 6513, 6514, 6515, 6516, 6517, 6518, 6519, 6520,
       6521, 6522, 6523, 6524, 6525, 6526, 6527, 6528, 6529, 6530, 6531,
       6532, 6533, 6534, 6535, 6536, 6537, 6538, 6539, 6540, 6541, 6542,
       6543, 6544, 6545, 6546, 6547, 6548, 6549, 6550, 6551, 6552, 6553,
       6554, 6555, 6556, 6557, 6558])

In [274]:
attribute_strategy_df.strategy_code.unique()

array(['BASE', 'TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ',
       'TORNADO_BASE:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ',
       'TORNADO_BASE:TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ',
       'TORNADO_BASE:TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ',
       'TORNADO_BASE:TX:ENTC:DEC_LOSSES_STRATEGY_NZ',
       'TORNADO_BASE:TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ',
       'TORNADO_BASE:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ',
       'TORNADO_BASE:TX:FGTV:DEC_LEAKS_STRATEGY_NZ',
       'TORNADO_BASE:TX:FGTV:INC_FLARE_STRATEGY_NZ',
       'TORNADO_BASE:TX:FRST:INCREASE_SEQUESTRATION_NZ',
       'TORNADO_BASE:TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ',
       'TORNADO_BASE:TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_CLINKER_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_HFCS_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_N2O_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_OTHER_FCS_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_PFCS_STRATEGY_NZ',

In [275]:
# Merge with cb_data on strategy_code
cb_data = cb_data.merge(attribute_strategy_df, on="strategy_code", how="left")
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6500
1,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6500
2,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6500
3,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6500
4,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6500


In [276]:
cb_data.strategy_code.unique()

array(['TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ',
       'TORNADO_BASE:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ',
       'TORNADO_BASE:TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ',
       'TORNADO_BASE:TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ',
       'TORNADO_BASE:TX:ENTC:DEC_LOSSES_STRATEGY_NZ',
       'TORNADO_BASE:TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ',
       'TORNADO_BASE:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ',
       'TORNADO_BASE:TX:FGTV:DEC_LEAKS_STRATEGY_NZ',
       'TORNADO_BASE:TX:FGTV:INC_FLARE_STRATEGY_NZ',
       'TORNADO_BASE:TX:FRST:INCREASE_SEQUESTRATION_NZ',
       'TORNADO_BASE:TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ',
       'TORNADO_BASE:TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_CLINKER_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_HFCS_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_N2O_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_OTHER_FCS_STRATEGY_NZ',
       'TORNADO_BASE:TX:IPPU:DEC_PFCS_STRATEGY_NZ',
       

In [277]:
cb_data.strategy_id.unique()

array([6500, 6501, 6502, 6503, 6504, 6505, 6506, 6507, 6508, 6509, 6510,
       6511, 6512, 6513, 6514, 6515, 6516, 6517, 6518, 6519, 6520, 6521,
       6522, 6523, 6524, 6525, 6526, 6527, 6528, 6529, 6530, 6531, 6532,
       6533, 6534, 6535, 6536, 6537, 6538, 6539, 6540, 6541, 6542, 6544,
       6545, 6546, 6547, 6548, 6549, 6550, 6551, 6552, 6553, 6554, 6555,
       6556, 6557, 6558])

In [278]:
# check for nans in strategy_id
cb_data[cb_data['strategy_id'].isna()]['strategy_code'].unique()

array([], dtype=object)

In [279]:
cb_data["sector"].unique()

array(['wali', 'entc', 'trns', 'lndu', 'waso', 'trww', 'lvst', 'agrc',
       'ccsq', 'inen', 'scoe', 'ippu', 'soil', 'lsmm', 'fgtv', 'pflo'],
      dtype=object)

In [280]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6500
1,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6500
2,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6500
3,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6500
4,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6500


In [281]:
# filter sectors
# target_sectors = ["wali", "trww", "waso", "soil", "ippu", "lvst", "agrc", "lndu", "lsmm"]
# cb_data = cb_data[cb_data["sector"].isin(target_sectors)].copy()
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6500
1,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6500
2,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6500
3,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6500
4,TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6500


In [282]:
# aggregate sum(value) grouped by strategy_id and cb_type
cb_data = (
    cb_data.groupby(["strategy_id", "cb_type"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head()

,strategy_id,cb_type,cumulative
0,6500,air_pollution,-4.739654e-18
1,6500,congestion,0.000000e+00
2,6500,crop_value,0.000000e+00
3,6500,ecosystem_services,0.000000e+00
4,6500,env_pollution,0.000000e+00


In [283]:
# unique cb_data types
cb_cats = cb_data["cb_type"].unique().tolist()

# long -> wide (R dcast equivalent)
wide_cb = (
    cb_data.pivot(index="strategy_id", columns="cb_type", values="cumulative")
      .reset_index()
)

# optional: remove column name from pivot for nicer printing
wide_cb.columns.name = None
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,6500,-4.739654e-18,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,-5.395756e-16,0.0,-0.214431,NaN,0.000000e+00
1,6501,1.118696e-02,0.0,NaN,-60.488145,1.449398,6.040279,-0.412602,0.0,0.0,0.075508,8.830576,0.0,3.238266e-03,0.0,-1.252058,159.082034,-2.885223e-07
2,6502,0.000000e+00,0.0,55.271272,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000e+00,0.0,0.000000,6.474635,0.000000e+00
3,6503,-9.236244e-02,0.0,NaN,145.939846,7.447893,0.000000,0.000000,0.0,0.0,0.291700,28.583300,0.0,-5.210304e-01,0.0,0.947703,NaN,1.933460e-07
4,6504,1.061802e-04,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,1.171610e-04,0.0,0.259754,NaN,0.000000e+00


In [284]:
cb_cats

['air_pollution',
 'congestion',
 'crop_value',
 'ecosystem_services',
 'env_pollution',
 'fuel_cost',
 'human_health',
 'ippu_value',
 'land_pollution',
 'lvst_value',
 'road_safety',
 'sector_specific',
 'system_cost',
 'technical_cost',
 'water_pollution',
 'technical_savings',
 'consumer_savings']

In [285]:
# --- 1) net_benefit = rowSums over all cb categories ---
wide_cb["net_benefit"] = wide_cb[cb_cats].sum(axis=1, skipna=True)

# --- 2) additional_benefits = rowSums excluding "technical_cost" ---
benefit_cols = [c for c in cb_cats if c != "technical_cost"]
wide_cb["additional_benefits"] = wide_cb[benefit_cols].sum(axis=1, skipna=True)

# --- 3) total_transformation_costs = rowSums over specific cols ---
cost_cols = ["technical_cost", "technical_savings", "fuel_cost"]

# (safe version: only use cols that exist in the df)
cost_cols = [c for c in cost_cols if c in wide_cb.columns]

wide_cb["total_transformation_costs"] = wide_cb[cost_cols].sum(axis=1, skipna=True)
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6500,-4.739654e-18,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,-5.395756e-16,0.0,-0.214431,NaN,0.000000e+00,-0.214431,-5.443153e-16,-0.214431
1,6501,1.118696e-02,0.0,NaN,-60.488145,1.449398,6.040279,-0.412602,0.0,0.0,...,8.830576,0.0,3.238266e-03,0.0,-1.252058,159.082034,-2.885223e-07,113.339415,1.145915e+02,157.417374
2,6502,0.000000e+00,0.0,55.271272,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,0.000000e+00,0.0,0.000000,6.474635,0.000000e+00,61.745906,6.174591e+01,6.474635
3,6503,-9.236244e-02,0.0,NaN,145.939846,7.447893,0.000000,0.000000,0.0,0.0,...,28.583300,0.0,-5.210304e-01,0.0,0.947703,NaN,1.933460e-07,182.597048,1.816493e+02,0.947703
4,6504,1.061802e-04,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.0,1.171610e-04,0.0,0.259754,NaN,0.000000e+00,0.259978,2.233412e-04,0.259754


## Merge emissions and cb data and save

In [286]:
tornado_emissions_agg_extended_df[tornado_emissions_agg_extended_df.strategy == "Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strategy TX:BASE"]

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
1,6500.0,760076.0,Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strate...,10815.341657,10817.757318,-2.4,AGRC,DEC_CH4_RICE


In [287]:
wide_cb.strategy_id.unique()

array([6500, 6501, 6502, 6503, 6504, 6505, 6506, 6507, 6508, 6509, 6510,
       6511, 6512, 6513, 6514, 6515, 6516, 6517, 6518, 6519, 6520, 6521,
       6522, 6523, 6524, 6525, 6526, 6527, 6528, 6529, 6530, 6531, 6532,
       6533, 6534, 6535, 6536, 6537, 6538, 6539, 6540, 6541, 6542, 6544,
       6545, 6546, 6547, 6548, 6549, 6550, 6551, 6552, 6553, 6554, 6555,
       6556, 6557, 6558])

In [288]:
print(wide_cb.shape)
print(tornado_emissions_agg_extended_df.shape)

(58, 21)
(59, 8)


In [289]:
df_merged = pd.merge(
    tornado_emissions_agg_extended_df,
    wide_cb,
    on="strategy_id",
    how="inner"
)

df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6500.0,760076.0,Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strate...,10815.341657,10817.757318,-2.4,AGRC,DEC_CH4_RICE,-4.739654e-18,0.0,...,0.000000,0.0,-5.395756e-16,0.0,-0.214431,NaN,0.000000e+00,-0.214431,-5.443153e-16,-0.214431
1,6501.0,770077.0,Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_N...,10654.457050,10817.757318,-163.3,AGRC,DEC_LOSSES_SUPPLY_CHAIN,1.118696e-02,0.0,...,8.830576,0.0,3.238266e-03,0.0,-1.252058,159.082034,-2.885223e-07,113.339415,1.145915e+02,157.417374
2,6502.0,780078.0,Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRAT...,10793.617644,10817.757318,-24.1,AGRC,INC_CONSERVATION_AGRICULTURE,0.000000e+00,0.0,...,0.000000,0.0,0.000000e+00,0.0,0.000000,6.474635,0.000000e+00,61.745906,6.174591e+01,6.474635
3,6503.0,790079.0,Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to St...,10495.590245,10817.757318,-322.2,AGRC,INC_PRODUCTIVITY,-9.236244e-02,0.0,...,28.583300,0.0,-5.210304e-01,0.0,0.947703,NaN,1.933460e-07,182.597048,1.816493e+02,0.947703
4,6504.0,800080.0,Add TX:ENTC:DEC_LOSSES_STRATEGY_NZ to Strategy...,10816.662339,10817.757318,-1.1,ENTC,DEC_LOSSES,1.061802e-04,0.0,...,0.000000,0.0,1.171610e-04,0.0,0.259754,NaN,0.000000e+00,0.259978,2.233412e-04,0.259754


In [290]:
print(df_merged.shape)

(58, 28)


### Below we have some hardcoded fixed exclusive of this study case to replace incorrect tranformation names

In [291]:
df_merged.columns

Index(['strategy_id', 'primary_id', 'strategy', 'emission_total',
       'base_emission_total', 'emission_diff', 'sector', 'transformation_name',
       'air_pollution', 'congestion', 'consumer_savings', 'crop_value',
       'ecosystem_services', 'env_pollution', 'fuel_cost', 'human_health',
       'ippu_value', 'land_pollution', 'lvst_value', 'road_safety',
       'sector_specific', 'system_cost', 'technical_cost', 'technical_savings',
       'water_pollution', 'net_benefit', 'additional_benefits',
       'total_transformation_costs'],
      dtype='object')

In [292]:
# multiply technical_cost by -1 to get positive costs
df_merged['technical_cost'] = df_merged['technical_cost'] * -1

# create marginal total abatement cost column
df_merged['marginal_total_abatement_cost_(USD/tCO2e)'] = (df_merged['technical_cost'] / df_merged['emission_diff'])*1000

# If technical_cost is positive then marginal_total_abatement_cost should be positive too.
# df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = np.where(df_merged["technical_cost"] > 0, df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs(), df_merged["marginal_total_abatement_cost_(USD/tCO2e)"])
df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs() * np.sign(df_merged["technical_cost"])


In [293]:
df_merged["strategy"].unique()

array(['Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:DEC_LOSSES_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FGTV:DEC_LEAKS_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FGTV:INC_FLARE_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:FRST:INCREASE_SEQUESTRATION_NZ to Strategy TX:BASE',
       'Add TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:IPPU:DEC_CLINKER_STRATEGY_NZ to Strategy TX:BASE',
       'Add TX:IPPU:DEC_HFCS_STRATEGY_NZ to Strategy TX:BASE',
       'Add T

In [294]:
df_merged['transformation_name_sector'] = df_merged['transformation_name'] + " - " + df_merged['sector']    

In [295]:
df_merged.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot.csv"), index=False)

### Create a QA version

In [296]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e),transformation_name_sector
0,6500.0,760076.0,Add TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ to Strate...,10815.341657,10817.757318,-2.4,AGRC,DEC_CH4_RICE,-4.739654e-18,0.0,...,-5.395756e-16,0.0,0.214431,NaN,0.000000e+00,-0.214431,-5.443153e-16,-0.214431,89.346080,DEC_CH4_RICE - AGRC
1,6501.0,770077.0,Add TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_N...,10654.457050,10817.757318,-163.3,AGRC,DEC_LOSSES_SUPPLY_CHAIN,1.118696e-02,0.0,...,3.238266e-03,0.0,1.252058,159.082034,-2.885223e-07,113.339415,1.145915e+02,157.417374,7.667228,DEC_LOSSES_SUPPLY_CHAIN - AGRC
2,6502.0,780078.0,Add TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRAT...,10793.617644,10817.757318,-24.1,AGRC,INC_CONSERVATION_AGRICULTURE,0.000000e+00,0.0,...,0.000000e+00,0.0,-0.000000,6.474635,0.000000e+00,61.745906,6.174591e+01,6.474635,0.000000,INC_CONSERVATION_AGRICULTURE - AGRC
3,6503.0,790079.0,Add TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ to St...,10495.590245,10817.757318,-322.2,AGRC,INC_PRODUCTIVITY,-9.236244e-02,0.0,...,-5.210304e-01,0.0,-0.947703,NaN,1.933460e-07,182.597048,1.816493e+02,0.947703,-2.941349,INC_PRODUCTIVITY - AGRC
4,6504.0,800080.0,Add TX:ENTC:DEC_LOSSES_STRATEGY_NZ to Strategy...,10816.662339,10817.757318,-1.1,ENTC,DEC_LOSSES,1.061802e-04,0.0,...,1.171610e-04,0.0,-0.259754,NaN,0.000000e+00,0.259978,2.233412e-04,0.259754,-236.140202,DEC_LOSSES - ENTC


In [297]:
df_merged.sector.unique()

array(['AGRC', 'ENTC', 'FGTV', 'FRST', 'INEN', 'IPPU', 'LNDU', 'LSMM',
       'LVST', 'PFLO', 'SCOE', 'SOIL', 'TRDE', 'TRNS', 'TRWW', 'WALI',
       'WASO'], dtype=object)

In [298]:
relevant_fields = [
    "transformation_name",
    "sector",
    "base_emission_total",
    "emission_total",
    "emission_diff",
    "technical_cost",
    "marginal_total_abatement_cost_(USD/tCO2e)"
]

# keep only relevant fields
df_merged_filtered = df_merged[relevant_fields]
df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
0,DEC_CH4_RICE,AGRC,10817.757318,10815.341657,-2.4,0.214431,89.346080
1,DEC_LOSSES_SUPPLY_CHAIN,AGRC,10817.757318,10654.457050,-163.3,1.252058,7.667228
2,INC_CONSERVATION_AGRICULTURE,AGRC,10817.757318,10793.617644,-24.1,-0.000000,0.000000
3,INC_PRODUCTIVITY,AGRC,10817.757318,10495.590245,-322.2,-0.947703,-2.941349
4,DEC_LOSSES,ENTC,10817.757318,10816.662339,-1.1,-0.259754,-236.140202


In [299]:
df_merged_filtered

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
0,DEC_CH4_RICE,AGRC,10817.757318,10815.341657,-2.4,2.144306e-01,8.934608e+01
1,DEC_LOSSES_SUPPLY_CHAIN,AGRC,10817.757318,10654.457050,-163.3,1.252058e+00,7.667228e+00
2,INC_CONSERVATION_AGRICULTURE,AGRC,10817.757318,10793.617644,-24.1,-0.000000e+00,0.000000e+00
3,INC_PRODUCTIVITY,AGRC,10817.757318,10495.590245,-322.2,-9.477028e-01,-2.941349e+00
4,DEC_LOSSES,ENTC,10817.757318,10816.662339,-1.1,-2.597542e-01,-2.361402e+02
5,TARGET_CLEAN_HYDROGEN,ENTC,10817.757318,10817.757318,-0.0,1.615378e-09,inf
6,TARGET_RENEWABLE_ELEC,ENTC,10817.757318,10778.621506,-39.1,-3.037530e+00,-7.768618e+01
7,DEC_LEAKS,FGTV,10817.757318,10806.317916,-11.4,1.143940e-01,1.003456e+01
8,INC_FLARE,FGTV,10817.757318,10817.698197,-0.1,5.912067e-04,5.912067e+00
9,INCREASE_SEQUESTRATION_NZ,FRST,10817.757318,10778.249249,-39.5,-1.454528e-11,-3.682349e-10


In [300]:
df_merged_filtered.to_clipboard(index=False)

In [301]:
df_merged_filtered.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_for_QA.csv"), index=False)